
# Step 1: Data Leakage & Shortcut Audit Playbook

This notebook helps you **rule out leakage or trivial shortcuts** behind unusually high accuracy in an image classification project (e.g., human vs. avatar vs. animal).  
It performs the following checks:

1. **Duplicate / near-duplicate images across splits** using perceptual hash (pHash) + optional SSIM refinement.
2. **Embedding-based near-duplicates** using a CNN feature extractor + cosine similarity.
3. **Identity leakage across splits (humans only)** using face embeddings + DBSCAN clustering.
4. **Source artifacts & shortcut signals** (e.g., logos, borders, resolution, source-domain predictability).

At the end, it produces CSV reports and suggests **remediation** (e.g., moving/removing images to fix split leakage).



## 0. Requirements

> Run this cell as-is or adapt to your environment. Some libraries (e.g., `face_recognition`) can be tricky to install; the notebook degrades gracefully and will skip identity checks if face embedding backends are unavailable.

```bash
# Core
pip install pillow imagehash scikit-image opencv-python-headless numpy pandas tqdm scikit-learn matplotlib

# Optional (for identity checks)
# One of the following is recommended (pick one):
# A) face_recognition (requires dlib; installation can be OS-specific)
# pip install face_recognition

# B) insightface (often easier on Linux; uses onnxruntime by default)
# pip install insightface onnxruntime

# Optional (for faster nearest neighbors on large sets)
# pip install faiss-cpu  # or faiss-gpu if you have CUDA
```



## 1. User Tunables & Project Layout

Fill these paths to match your project. The notebook works in two modes:

- **Directory mode**: expects a structure like `data/{train,val,test}/{class}/image.jpg`.
- **CSV mode**: provide a CSV with columns: `path,label,split` and optional `source,person_hint`.


In [9]:
%pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.7 MB/s eta 0:00:00


In [10]:

# ===== USER TUNABLES =====
from pathlib import Path

# Choose one:
DATA_ROOT = Path("data/final")          # Directory mode root (e.g., data/train, data/val, data/test)
METADATA_CSV = None               # Or: Path("metadata.csv")  # CSV mode

# Output directory for reports & caches
OUT_DIR = Path("audit_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Perceptual-hash duplicate settings
PHASH_SIZE = 16                   # ImageHash hash size (phash); 16 => 64-bit
HAMMING_THRESHOLD = 5             # Max Hamming distance to consider 'near-duplicate'

# Embedding-based duplicate settings
EMBED_MODEL = "EfficientNetB0"    # Keras extractor backbone (EfficientNetB0 or ResNet50)
EMBED_BATCH = 32
EMBED_SIM_THRESHOLD = 0.985       # Cosine similarity threshold for near-dup pairs

# Identity leakage (humans only)
RUN_IDENTITY_CHECKS = True        # Set False to skip if deps are problematic
FACE_BACKEND = "auto"             # "face_recognition" | "insightface" | "auto"
DBSCAN_EPS = 0.5                  # Clustering tightness for identity groups
DBSCAN_MIN_SAMPLES = 2

# Source artifact checks
EXPECT_SOURCE_COLUMN = True       # If using CSV mode and you have 'source' metadata
RANDOM_SEED = 42
# ==========================



## 2. Load Image Index

Build a unified DataFrame with at least: `path, split, label`, and optionally `source`.


In [11]:

import os
import pandas as pd
from typing import List, Dict
from pathlib import Path

def list_images(root: Path, splits=("train","val","test")) -> pd.DataFrame:
    rows = []
    for split in splits:
        split_dir = root / split
        if not split_dir.exists():
            continue
        for cls in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            for img in cls.rglob("*"):
                if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                    rows.append({
                        "path": str(img.as_posix()),
                        "split": split,
                        "label": cls.name,
                        "source": cls.parent.name if False else None  # placeholder
                    })
    df = pd.DataFrame(rows)
    return df

if METADATA_CSV:
    df = pd.read_csv(METADATA_CSV)
    # Normalize columns
    needed = {"path","label","split"}
    if not needed.issubset(set(df.columns)):
        raise ValueError(f"CSV must have columns {needed}")
    # Ensure str paths
    df["path"] = df["path"].astype(str)
else:
    df = list_images(DATA_ROOT)

# Basic hygiene
df = df.drop_duplicates(subset=["path"]).reset_index(drop=True)
df["path"] = df["path"].apply(lambda p: str(Path(p)))
if "source" not in df.columns:
    df["source"] = None

print(df.head())
print("Counts by split/label:\n", df.groupby(["split","label"]).size())
df.to_csv(OUT_DIR / "image_index.csv", index=False)


                                           path  split   label source
0  data/final/train/animal/a5b067d023ca6582.jpg  train  animal   None
1  data/final/train/animal/44935642e882041b.jpg  train  animal   None
2  data/final/train/animal/46c81d173da7e995.jpg  train  animal   None
3  data/final/train/animal/4494d8932021afa0.jpg  train  animal   None
4  data/final/train/animal/4512ede202084b7f.jpg  train  animal   None
Counts by split/label:
 split  label 
test   animal    1000
       avatar    1000
       human     1000
train  animal    8000
       avatar    8000
       human     8000
val    animal    1000
       avatar    1000
       human     1000
dtype: int64



## 3. Duplicate / Near-Duplicate Scan (Perceptual Hash)

We compute **pHash** for all images, then find **cross-split** collisions within a Hamming distance threshold.
To refine, you can optionally compute **SSIM** for candidate pairs.


In [12]:
from PIL import Image, UnidentifiedImageError
import imagehash
import numpy as np
from tqdm import tqdm

def compute_phash(path: str, hash_size=16) -> str:
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")
            h = imagehash.phash(im, hash_size=hash_size)
            # Return hash as string
            return str(h)
    except Exception as e:
        return None

# Compute or load cache
phash_cache_path = OUT_DIR / "phashes.parquet"
if phash_cache_path.exists():
    phash_df = pd.read_parquet(phash_cache_path)
    # Ensure phash_str is string type
    phash_df['phash_str'] = phash_df['phash_str'].astype(str)
else:
    phash_vals = []
    for p in tqdm(df["path"], desc="Computing pHash"):
        phash_vals.append(compute_phash(p, hash_size=PHASH_SIZE))
    phash_df = df.copy()
    phash_df["phash_str"] = phash_vals
    # Save the string representation
    phash_df.to_parquet(phash_cache_path, index=False)


# Bucket by leading bits to avoid O(N^2)
def leading_bits_str(hash_str: str, bits: int = 16) -> int:
    if hash_str is None or not hash_str:
        return -1
    try:
        # Convert first 'bits' number of hex characters to an integer
        # Each hex char is 4 bits, so we need ceil(bits/4) characters
        num_chars = (bits + 3) // 4
        return int(hash_str[:num_chars], 16)
    except ValueError:
        return -1 # Handle cases where conversion fails

phash_df["bucket16"] = phash_df["phash_str"].apply(lambda v: leading_bits_str(v, bits=16) if v is not None else -1)


def hamming_distance_hex_str(hex_str_a: str, hex_str_b: str) -> int:
    if hex_str_a is None or hex_str_b is None:
        return float('inf') # Or some other indicator of invalid comparison

    try:
        # Convert hex strings to large integers and compute hamming distance
        int_a = int(hex_str_a, 16)
        int_b = int(hex_str_b, 16)
        return bin(int_a ^ int_b).count("1")
    except ValueError:
         return float('inf') # Handle cases where conversion fails


# Compare within buckets, only cross-split
pairs = []
# Group by the bucket, iterate through paths and their string hashes
for b, grp in phash_df.dropna(subset=["phash_str"]).groupby("bucket16"):
    arr = grp[["path","split","label","phash_str"]].to_numpy()
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            if arr[i][1] == arr[j][1]:
                continue  # we only care about cross-split duplicates for leakage
            # Pass the hex strings to the hamming distance function
            d = hamming_distance_hex_str(arr[i][3], arr[j][3])
            if d != float('inf') and d <= HAMMING_THRESHOLD:
                pairs.append((arr[i][0], arr[i][1], arr[i][2], arr[j][0], arr[j][1], arr[j][2], d))


dup_phash_df = pd.DataFrame(pairs, columns=[
    "path_a","split_a","label_a","path_b","split_b","label_b","hamming"
]).sort_values("hamming")

dup_phash_df.to_csv(OUT_DIR / "cross_split_phash_near_duplicates.csv", index=False)
print("Found near-duplicate cross-split pairs (pHash):", len(dup_phash_df))
dup_phash_df.head()

Computing pHash: 100%|██████████| 30000/30000 [00:42<00:00, 710.27it/s]


Found near-duplicate cross-split pairs (pHash): 4


,path_a,split_a,label_a,path_b,split_b,label_b,hamming
0,data/final/train/avatar/4c5f4afa8aa9652c.jpg,train,avatar,data/final/val/avatar/cdf194dd33602883.jpg,val,avatar,4
1,data/final/train/avatar/49e8a354b427c17a.jpg,train,avatar,data/final/test/avatar/badb23cf66a87779.jpg,test,avatar,4
2,data/final/train/avatar/12705e980591be21.jpg,train,avatar,data/final/test/avatar/f444eb067aad6613.jpg,test,avatar,4
3,data/final/train/avatar/5c04c3ce8ba8fd7b.jpg,train,avatar,data/final/val/avatar/3f9f12245edf73c2.jpg,val,avatar,4


In [13]:

# Optional: refine candidates via SSIM to reduce false positives
from skimage.metrics import structural_similarity as ssim
import cv2

def load_gray_resized(path: str, size=256):
    im = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if im is None:
        # Fallback to PIL
        with Image.open(path) as pil:
            pil = pil.convert("RGB").resize((size,size))
            im = np.array(pil)[:, :, ::-1]  # RGB->BGR
    im = cv2.resize(im, (size,size), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    return gray

refined = []
for idx, row in tqdm(dup_phash_df.iterrows(), total=len(dup_phash_df), desc="SSIM refine"):
    try:
        a = load_gray_resized(row["path_a"])
        b = load_gray_resized(row["path_b"])
        s = ssim(a, b, data_range=255)
        refined.append(s)
    except Exception:
        refined.append(None)

dup_phash_df["ssim"] = refined
dup_phash_df.to_csv(OUT_DIR / "cross_split_phash_near_duplicates_ssim.csv", index=False)
dup_phash_df.head()


SSIM refine: 100%|██████████| 4/4 [00:00<00:00, 58.76it/s]


,path_a,split_a,label_a,path_b,split_b,label_b,hamming,ssim
0,data/final/train/avatar/4c5f4afa8aa9652c.jpg,train,avatar,data/final/val/avatar/cdf194dd33602883.jpg,val,avatar,4,0.964917
1,data/final/train/avatar/49e8a354b427c17a.jpg,train,avatar,data/final/test/avatar/badb23cf66a87779.jpg,test,avatar,4,0.971597
2,data/final/train/avatar/12705e980591be21.jpg,train,avatar,data/final/test/avatar/f444eb067aad6613.jpg,test,avatar,4,0.945609
3,data/final/train/avatar/5c04c3ce8ba8fd7b.jpg,train,avatar,data/final/val/avatar/3f9f12245edf73c2.jpg,val,avatar,4,0.919868



## 4. Embedding-Based Near-Duplicates (CNN Features + Cosine)

This section extracts CNN features (from EfficientNetB0 or ResNet50) and finds **cross-split** near-duplicates by cosine similarity.


In [14]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import applications, Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tqdm import tqdm

# Build feature extractor
IMG_SIZE = 224
if EMBED_MODEL.lower() == "resnet50":
    base = applications.ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
    preprocess = applications.resnet50.preprocess_input
elif EMBED_MODEL.lower() == "efficientnetb0":
    base = applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
    preprocess = applications.efficientnet.preprocess_input
else:
    raise ValueError("Unsupported EMBED_MODEL")

x = GlobalAveragePooling2D()(base.output)
extractor = Model(base.input, x)

def load_and_preprocess(path: str, size=IMG_SIZE):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (size, size), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    img = preprocess(img)
    return img

# Build datasets by split to only compare cross-split
paths_train = df[df["split"]=="train"]["path"].tolist()
paths_val   = df[df["split"]=="val"]["path"].tolist()
paths_test  = df[df["split"]=="test"]["path"].tolist()

def batch_embeddings(paths):
    embs = []
    for i in tqdm(range(0, len(paths), EMBED_BATCH), desc="Embedding"):
        batch = paths[i:i+EMBED_BATCH]
        imgs = tf.stack([load_and_preprocess(p) for p in batch], axis=0)
        v = extractor.predict(imgs, verbose=0)
        embs.append(v)
    if embs:
        return np.vstack(embs)
    else:
        return np.zeros((0, x.shape[-1]))

emb_train = batch_embeddings(paths_train)
emb_val   = batch_embeddings(paths_val)
emb_test  = batch_embeddings(paths_test)

# Cosine similarity function
from sklearn.metrics.pairwise import cosine_similarity

def cross_pairs(paths_a, emb_a, paths_b, emb_b, threshold):
    pairs = []
    # Use blockwise computation to control memory
    block = 2048
    for i in tqdm(range(0, len(paths_a), block), desc="Cosine blocks A->B"):
        eA = emb_a[i:i+block]
        sims = cosine_similarity(eA, emb_b)
        for ii, row in enumerate(sims):
            jj = np.where(row >= threshold)[0]
            for j in jj:
                pairs.append((paths_a[i+ii], paths_b[j], float(row[j])))
    return pairs

pairs_train_val = cross_pairs(paths_train, emb_train, paths_val, emb_val, EMBED_SIM_THRESHOLD)
pairs_train_test = cross_pairs(paths_train, emb_train, paths_test, emb_test, EMBED_SIM_THRESHOLD)
pairs_val_test   = cross_pairs(paths_val,   emb_val,   paths_test, emb_test, EMBED_SIM_THRESHOLD)

emb_pairs = pd.DataFrame(pairs_train_val + pairs_train_test + pairs_val_test, columns=["path_a","path_b","cos_sim"])

# Join split/label info
meta = df.set_index("path")[["split","label"]]
emb_pairs = (emb_pairs
             .join(meta, on="path_a")
             .rename(columns={"split":"split_a","label":"label_a"})
             .join(meta, on="path_b")
             .rename(columns={"split":"split_b","label":"label_b"}))

# Keep only cross-split
emb_pairs = emb_pairs[emb_pairs["split_a"] != emb_pairs["split_b"]].sort_values("cos_sim", ascending=False)

emb_pairs.to_csv(OUT_DIR / "cross_split_embedding_near_duplicates.csv", index=False)
print("Embedding-based cross-split near-duplicates:", len(emb_pairs))
emb_pairs.head()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Cosine blocks A->B: 100%|██████████| 2/2 [00:00<00:00, 17.37it/s]

Embedding-based cross-split near-duplicates: 25


,path_a,path_b,cos_sim,split_a,label_a,split_b,label_b
24,data/final/val/avatar/e31ef2b81371290c.jpg,data/final/test/avatar/17e596e523b055db.jpg,0.994147,val,avatar,test,avatar
16,data/final/train/avatar/3a576c7cd97ff037.jpg,data/final/test/avatar/6510af2cc23316fc.jpg,0.993596,train,avatar,test,avatar
1,data/final/train/avatar/7fe9ac0ee636bc03.jpg,data/final/val/avatar/0b74defdf58430db.jpg,0.991381,train,avatar,val,avatar
8,data/final/train/avatar/922ab687e688d601.jpg,data/final/val/avatar/43510660773c1d26.jpg,0.991149,train,avatar,val,avatar
10,data/final/train/human/2df519d606557f24.jpg,data/final/val/human/4e6a555ab81a533c.jpg,0.990577,train,human,val,human



## 5. Identity Leakage Across Splits (Humans Only)

Detect if the **same person** appears across train/val/test. Two options:

- `face_recognition` (128D embeddings; easy API but platform-dependent install)
- `insightface` (strong performer; works with `onnxruntime` CPU or GPU)

If neither backend is available or `RUN_IDENTITY_CHECKS=False`, this section will skip.


In [15]:

import importlib

def try_import(mod):
    try:
        return importlib.import_module(mod)
    except Exception:
        return None

def pick_face_backend(pref: str = FACE_BACKEND):
    if pref == "face_recognition":
        return "face_recognition" if try_import("face_recognition") else None
    if pref == "insightface":
        return "insightface" if try_import("insightface") else None
    # auto
    if try_import("face_recognition"):
        return "face_recognition"
    if try_import("insightface"):
        return "insightface"
    return None

backend = pick_face_backend()
print("Face backend:", backend)

if RUN_IDENTITY_CHECKS and backend is not None:
    human_df = df[df["label"].str.lower().isin(["human","person","people","face","faces"]) | (df["label"].str.lower()=="human_face")].copy()
    if human_df.empty:
        print("No obvious 'human' label detected; skipping identity checks.")
        identity_report = pd.DataFrame()
    else:
        import numpy as np
        from tqdm import tqdm
        import cv2

        def read_rgb(path):
            img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                with Image.open(path) as pil:
                    pil = pil.convert("RGB")
                    img = np.array(pil)[:, :, ::-1]  # RGB->BGR
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if backend == "face_recognition":
            import face_recognition
            def face_embed(img_rgb):
                boxes = face_recognition.face_locations(img_rgb, model="hog")
                if not boxes:
                    return None
                encs = face_recognition.face_encodings(img_rgb, boxes)
                return encs[0] if encs else None
        else:
            import insightface
            from insightface.app import FaceAnalysis
            app = FaceAnalysis(name="buffalo_l")
            app.prepare(ctx_id=0 if tf.config.list_physical_devices('GPU') else -1, det_size=(640,640))
            def face_embed(img_rgb):
                faces = app.get(img_rgb)
                if not faces:
                    return None
                return faces[0].normed_embedding.astype("float32")

        embeds = []
        for p in tqdm(human_df["path"], desc="Face embeddings (humans)"):
            try:
                im = read_rgb(p)
                e = face_embed(im)
            except Exception:
                e = None
            embeds.append(e)

        human_df["face_emb"] = embeds
        valid = human_df[human_df["face_emb"].notna()].copy()
        if valid.empty:
            print("No face embeddings extracted; skipping identity clustering.")
            identity_report = pd.DataFrame()
        else:
            X = np.vstack(valid["face_emb"].to_list())
            # Cluster identities
            from sklearn.cluster import DBSCAN
            db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="euclidean").fit(X)
            valid["identity_cluster"] = db.labels_

            # Any clusters spread across multiple splits?
            leak_rows = []
            for cid, grp in valid.groupby("identity_cluster"):
                if cid == -1:  # noise
                    continue
                splits = grp["split"].unique().tolist()
                if len(splits) > 1:
                    for row in grp.itertuples(index=False):
                        leak_rows.append((cid, row.path, row.split, row.label))
            identity_report = pd.DataFrame(leak_rows, columns=["cluster","path","split","label"])

        identity_report.to_csv(OUT_DIR / "cross_split_identity_leakage.csv", index=False)
        print(f"Identity leakage rows: {len(identity_report)}")
else:
    print("Skipping identity checks (deps unavailable or disabled).")


Face backend: None
Skipping identity checks (deps unavailable or disabled).



## 6. Source Artifact & Shortcut Checks

This section looks for dataset artifacts that might act as shortcuts:
- Correlations between **source domain** and **label**
- Predictability of label **using only metadata** (e.g., source, resolution, file size)
- Basic border/watermark heuristics


In [16]:

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def fast_image_stats(path):
    try:
        im_bytes = np.fromfile(path, dtype=np.uint8)
        size_kb = im_bytes.size / 1024.0
        im = cv2.imdecode(im_bytes, cv2.IMREAD_COLOR)
        if im is None:
            return None
        h, w = im.shape[:2]
        # Brightness & contrast proxies
        gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        mean = float(gray.mean())
        std = float(gray.std())
        # Simple black-border ratio (edges close to 0)
        border_w = max(1, int(0.02 * min(h,w)))
        edges = np.concatenate([
            gray[:border_w, :].flatten(),
            gray[-border_w:, :].flatten(),
            gray[:, :border_w].flatten(),
            gray[:, -border_w:].flatten()
        ])
        black_border_ratio = float((edges < 5).mean())
        return dict(width=w, height=h, size_kb=size_kb, mean=mean, std=std, black_border_ratio=black_border_ratio)
    except Exception:
        return None

stats_cache = OUT_DIR / "image_stats.parquet"
if stats_cache.exists():
    stats_df = pd.read_parquet(stats_cache)
else:
    rows = []
    for p in tqdm(df["path"], desc="Image stats"):
        st = fast_image_stats(p)
        if st is None:
            continue
        st.update({"path": p})
        rows.append(st)
    stats_df = pd.DataFrame(rows)
    stats_df.to_parquet(stats_cache, index=False)

meta_df = df.merge(stats_df, on="path", how="left")

# 6a) Source vs Label crosstab
if "source" in meta_df.columns and meta_df["source"].notna().any():
    ctab = pd.crosstab(meta_df["source"], meta_df["label"])
    ctab.to_csv(OUT_DIR / "source_label_crosstab.csv")
    display(ctab.head())

# 6b) Predict label using metadata-only features
features = ["width","height","size_kb","mean","std","black_border_ratio"]
if "source" in meta_df.columns and meta_df["source"].notna().any():
    features.append("source")

use_df = meta_df.dropna(subset=["label"] + features).copy()
X = use_df[features]
y = use_df["label"]

cat = ["source"] if "source" in features else []
num = [f for f in features if f not in cat]

pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

clf = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=1000, multi_class="auto"))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)
clf.fit(Xtr, ytr)
yp = clf.predict(Xte)
rep = classification_report(yte, yp, output_dict=False)
print(rep)

with open(OUT_DIR / "metadata_only_shortcut_report.txt","w") as f:
    f.write(rep)


Image stats: 100%|██████████| 30000/30000 [00:23<00:00, 1259.53it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

      animal       0.70      0.74      0.72      2500
      avatar       1.00      1.00      1.00      2500
       human       0.72      0.68      0.70      2500

    accuracy                           0.81      7500
   macro avg       0.81      0.81      0.80      7500
weighted avg       0.81      0.81      0.80      7500




## 7. Remediation Plan (Automatable Steps)

- **Cross-split duplicates (pHash/embeddings):**
  - Prefer **keeping** the item in **train** and **removing** from **val/test**.
  - Pick the earliest file or the one with highest resolution when deduplicating within split.

- **Identity leakage across splits:**
  - For any cluster that spans multiple splits, consolidate all images to a **single split** (ideally train) or **remove** extras from val/test.

- **Source artifacts:**
  - If `source → label` is highly predictive (e.g., metadata-only LR > ~0.75 accuracy), either **re-balance by source** or **mask artifacts** (e.g., random cropping that removes borders, blur watermarks).

Run the cell below to **generate an `exclusions.csv`** with suggested removals (non-destructive). Review before applying.


In [17]:

import pandas as pd

# Load previously computed reports (if present)
exclusions = []

# 1) pHash-based cross-split near-duplicates
phash_report = OUT_DIR / "cross_split_phash_near_duplicates_ssim.csv"
if phash_report.exists():
    ph = pd.read_csv(phash_report)
else:
    ph = pd.read_csv(OUT_DIR / "cross_split_phash_near_duplicates.csv")
if not ph.empty:
    # Strategy: if one of the pair is in val/test, mark it for exclusion; prefer keeping 'train'
    for row in ph.itertuples(index=False):
        keep = None
        drop = None
        if row.split_a == "train" and row.split_b != "train":
            drop = row.path_b
        elif row.split_b == "train" and row.split_a != "train":
            drop = row.path_a
        else:
            # Neither is train; drop the one from 'test' by default
            drop = row.path_b if row.split_b == "test" else row.path_a
        if drop:
            exclusions.append(("phash_dup", drop))

# 2) Embedding-based cross-split near-duplicates
emb_report = OUT_DIR / "cross_split_embedding_near_duplicates.csv"
if Path(emb_report).exists():
    emb = pd.read_csv(emb_report)
    for row in emb.itertuples(index=False):
        # same policy as above
        if row.split_a == "train" and row.split_b != "train":
            drop = row.path_b
        elif row.split_b == "train" and row.split_a != "train":
            drop = row.path_a
        else:
            drop = row.path_b if row.split_b == "test" else row.path_a
        exclusions.append(("emb_dup", drop))

# 3) Identity leakage clusters
id_report = OUT_DIR / "cross_split_identity_leakage.csv"
if Path(id_report).exists():
    ident = pd.read_csv(id_report)
    # For each identity cluster, keep only items in the preferred split (train) and drop others
    if not ident.empty:
        pref_split = "train"
        for cluster, grp in ident.groupby("cluster"):
            if pref_split in set(grp["split"]):
                for row in grp.itertuples(index=False):
                    if row.split != pref_split:
                        exclusions.append(("identity_leak", row.path))

excl_df = pd.DataFrame(exclusions, columns=["reason","path"]).drop_duplicates()
excl_df.to_csv(OUT_DIR / "exclusions.csv", index=False)
print(f"Suggested exclusions: {len(excl_df)}")
excl_df.head()


Suggested exclusions: 27


,reason,path
0,phash_dup,data/final/val/avatar/cdf194dd33602883.jpg
1,phash_dup,data/final/test/avatar/badb23cf66a87779.jpg
2,phash_dup,data/final/test/avatar/f444eb067aad6613.jpg
3,phash_dup,data/final/val/avatar/3f9f12245edf73c2.jpg
4,emb_dup,data/final/test/avatar/17e596e523b055db.jpg



## 8. (Optional) Apply Exclusions Non-Destructively

This cell writes all excluded files' **relative paths** to `exclusions.txt`. You can modify your data loader to **skip any path in this list** without moving files around.


In [18]:

excl = pd.read_csv(OUT_DIR / "exclusions.csv")
txt_path = OUT_DIR / "exclusions.txt"
txt_path.write_text("\n".join(sorted(excl["path"].tolist())))
print(f"Wrote {txt_path}")


Wrote audit_outputs/exclusions.txt



## 9. Summary Checklist

- [ ] pHash duplicate report reviewed (`cross_split_phash_near_duplicates*.csv`)
- [ ] Embedding-based near-duplicate report reviewed (`cross_split_embedding_near_duplicates.csv`)
- [ ] Identity leakage report reviewed (`cross_split_identity_leakage.csv`)
- [ ] Metadata-only shortcut report reviewed (`metadata_only_shortcut_report.txt`)
- [ ] `exclusions.csv` validated and applied to data loader (skip listed images in val/test)
- [ ] Re-train baseline and re-check metrics/curves after remediation


# Tools

In [19]:
# !gcloud auth application-default login

In [20]:
# import os

# # Specify the source bucket and destination local path
# source_bucket = "gs://along-capstone-data"
# destination_path = "/content/capstone_data/"

# # Create the destination directory if it doesn't exist
# os.makedirs(destination_path, exist_ok=True)

# # Copy files using gsutil
# !gsutil -m cp -r "$source_bucket" "$destination_path"

# print(f"Files copied from {source_bucket} to {destination_path}")

In [ ]:
bucket_name = 'gs://along-capstone-data'
source_directory = 'final'
destination_directory = 'data'

# Create the destination directory if it doesn't exist
import os
os.makedirs(destination_directory, exist_ok=True)

import json
from google.colab import userdata

# Get the service account key from Colab Secrets
service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

# Define the path to save the service account key file
key_file_path = 'service_account_key.json'

# Save the service account key to a file
with open(key_file_path, 'w') as f:
    json.dump(service_account_info, f)

# Authenticate gcloud and gsutil using the service account key file
!gcloud auth activate-service-account --key-file {key_file_path}

# Download files using gsutil with multiple threads
# -m enables multithreading
# -r recursively copies directories and files
!gsutil -m cp -r {bucket_name}/{source_directory} {destination_directory}

print("Download complete.")

Streaming output truncated to the last 5000 lines.
Copying gs://along-capstone-data/final/train/human/a2e6c1f0870fb027.jpg...
Copying gs://along-capstone-data/final/train/human/a32024930ce7a8f1.jpg...
Copying gs://along-capstone-data/final/train/human/a31dd373e0f49712.jpg...
Copying gs://along-capstone-data/final/train/human/a35ce1aa4387d2ed.jpg...
Copying gs://along-capstone-data/final/train/human/a26745c7af3b3986.jpg...
Copying gs://along-capstone-data/final/train/human/a3a91bd7476e7096.jpg...
Copying gs://along-capstone-data/final/train/human/a2d1ab2fd3f1f006.jpg...
Copying gs://along-capstone-data/final/train/human/a23002c59fcebb52.jpg...
Copying gs://along-capstone-data/final/train/human/a24c785aea6ff36a.jpg...
Copying gs://along-capstone-data/final/train/human/a24d29ec03965285.jpg...
Copying gs://along-capstone-data/final/train/human/a2c8774cb111a0ac.jpg...
Copying gs://along-capstone-data/final/train/human/a2bb140e38f09ab8.jpg...
Copying gs://along-capstone-data/final/train/huma